# White matter hyperintensities: tutorial T1/FLAIR pair

Align T1 to FLAIR and run the pretrained SYSU-Media model using the same acquired
pair as the ANTsX tutorial (Figshare files 40251796 and 40251793). Both images
download and are cached on first use. Override the defaults with absolute paths
or `ANTSTORCH_EXAMPLE_T1` and `ANTSTORCH_EXAMPLE_FLAIR`; specify both to avoid
mixing subjects. Model weights also download on first use. Registration uses
ANTsPy on CPU; WMH inference uses ANTsTorch on the selected device.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
from time import perf_counter
_example_started = perf_counter()

import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

# Synchronize the selected accelerator so timings include completed GPU work.
from contextlib import contextmanager
_example_operation_times = []

def _synchronize_example_device():
    selected = torch.device(device)
    if selected.type == "cuda":
        torch.cuda.synchronize(selected)
    elif selected.type == "mps":
        torch.mps.synchronize()

@contextmanager
def time_step(label):
    _synchronize_example_device()
    started = perf_counter()
    try:
        yield
    finally:
        _synchronize_example_device()
        seconds = perf_counter() - started
        _example_operation_times.append((label, seconds))
        print(f"{label}: {seconds:.4f} s")

In [ ]:
import os
from pathlib import Path
t1_path = os.environ.get("ANTSTORCH_EXAMPLE_T1")
flair_path = os.environ.get("ANTSTORCH_EXAMPLE_FLAIR")
# Alternatively: t1_path = "/absolute/path/t1.nii.gz"
#              flair_path = "/absolute/path/flair.nii.gz"
if (t1_path is None) != (flair_path is None):
    raise ValueError("Specify both T1 and FLAIR paths, or neither to use the tutorial pair.")
if t1_path is None:
    t1_path = antstorch.get_antstorch_data("tutorialWmhT1")
    flair_path = antstorch.get_antstorch_data("tutorialWmhFlair")
for name, path in [("T1", t1_path), ("FLAIR", flair_path)]:
    if not Path(path).is_file():
        raise FileNotFoundError(f"{name} image does not exist: {path}")
t1 = ants.image_read(str(t1_path)).clone("float")
flair = ants.image_read(str(flair_path)).clone("float")
for image in (t1, flair):
    assert image.dimension == 3 and not image.has_components
    assert np.isfinite(image.numpy()).all() and np.ptp(image.numpy()) > 0

## Align modalities rather than copying their headers
Rigidly align the T1 to the FLAIR reference grid with ANTsPy. If the images are
already anatomically aligned, set `already_aligned=True`; resampling alone
changes the grid and does not estimate alignment. Inspect the overlay either way.

In [ ]:
already_aligned = False
output_dir = Path("example_outputs/white_matter_hyperintensities")
output_dir.mkdir(parents=True, exist_ok=True)
if already_aligned:
    aligned_t1 = ants.resample_image_to_target(t1, flair, interp_type="linear")
else:
    registration = ants.registration(
        fixed=flair, moving=t1, type_of_transform="Rigid", random_seed=42,
        outprefix=str(output_dir / "t1_to_flair_"),
    )
    aligned_t1 = registration["warpedmovout"]
assert aligned_t1.shape == flair.shape
for attribute in ("spacing", "origin", "direction"):
    np.testing.assert_allclose(getattr(aligned_t1, attribute), getattr(flair, attribute))
ants.plot(flair, overlay=aligned_t1, axis=2, nslices=6, overlay_alpha=0.4)

In [ ]:
with time_step('WMH segmentation (probability)'):
    probability = antstorch.sysu_media_wmh_segmentation(
        flair, t1=aligned_t1, use_ensemble=True, device=device, verbose=True,
    )
values = probability.numpy()
assert probability.shape == flair.shape and np.isfinite(values).all()
assert values.min() >= -1e-6 and values.max() <= 1 + 1e-6
for attribute in ("spacing", "origin", "direction"):
    np.testing.assert_allclose(getattr(probability, attribute), getattr(flair, attribute))
threshold = 0.5  # Demonstration threshold; tune/validate for the intended data.
mask = ants.threshold_image(probability, threshold, 1., 1, 0)
ants.plot(flair, overlay=mask, axis=2, nslices=8,
          overlay_cmap="Reds", overlay_alpha=0.6)
ants.image_write(aligned_t1, str(output_dir / "t1_in_flair.nii.gz"))
ants.image_write(probability, str(output_dir / "wmh_probability.nii.gz"))
ants.image_write(mask, str(output_dir / "wmh_mask.nii.gz"))
volume = float(np.count_nonzero(mask.numpy()) * np.prod(flair.spacing))
print(f"Thresholded volume at {threshold}: {volume:.1f} cubic physical units")
print("Outputs (matching files are overwritten):", output_dir.resolve())

The model returns a probability image, not an automatically validated lesion
burden. Inspect nonbrain signal, periventricular regions and artifacts. Brain
or white-matter masking is a separate choice; the wrapper leaves such
postprocessing to the caller. T1 can be omitted from the function call for
its FLAIR-only model, but that is a different inference configuration.

## Execution time

Run **Restart Kernel and Run All Cells** for a comparable end-to-end duration.
The total starts at the first setup cell and includes imports, downloads, data
preparation, computation, plotting and file writes. Kernel startup is excluded;
manual pauses between cells are included. Operation timings include any
preprocessing or asset downloads performed inside the called function. First
runs can be slower because caches are empty. Timings describe this run on the
selected device, not a fixed performance estimate for other machines.

In [ ]:
_synchronize_example_device()
total_seconds = perf_counter() - _example_started
operation_seconds = sum(seconds for _, seconds in _example_operation_times)
print(f"Total elapsed time: {total_seconds:.2f} s ({total_seconds / 60:.2f} min)")
print(f"Timed operations: {operation_seconds:.4f} s on {device}")